# Projet NoSQL & Big Data - Films TMDB (Cassandra + Redis)

## Contexte

Ce notebook est mon carnet de bord pour le projet final NoSQL. Je débute sur ces technos, donc j'explique au fur et à mesure ce que je découvre (les commandes, les concepts, pourquoi je fais tel ou tel choix).

## Le choix de la base de données

L'énoncé du projet propose comme exemples MongoDB (orienté document) ou Neo4j (orienté graphe). Avec l'accord de mon formateur, j'ai choisi de travailler sur deux autres bases NoSQL pour comparer leurs usages :

- **Cassandra** (orientée colonnes larges) pour stocker toutes les fiches films de façon durable, organisées selon plusieurs axes d'interrogation (par genre, par réalisateur, par acteur, par année).
- **Redis** (clé-valeur) pour représenter un état "rapide à consulter" : classement des films les mieux notés, fiche film en cache.

## Le jeu de données

J'ai testé plusieurs jeux de données avant celui-ci (un journal de clics e-commerce, puis un dataset de notation de films MovieLens), avant que mon formateur me demande d'utiliser un dataset avec plus de colonnes pour montrer des requêtes plus riches. J'ai atterri sur **TMDB 5000** (public, disponible sur Kaggle) : les métadonnées de 4803 films, plus le casting/l'équipe technique.

Deux fichiers :
- `tmdb_5000_movies.csv` : budget, genres, langue, popularité, revenu, durée, note, nombre de votes...
- `tmdb_5000_credits.csv` : le casting (acteurs) et l'équipe (réalisateur...) de ces mêmes films

## 01. Exploration du dataset brut


In [14]:
import pandas as pd
import ast
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

movies = pd.read_csv("../tmdb_5000_movies.csv")
credits = pd.read_csv("../tmdb_5000_credits.csv")
print(movies.shape)
print(credits.shape)
credits.head(3)


(4803, 20)
(4803, 4)


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


### Un premier coup d'oeil

Je regarde à quoi ressemblent les colonnes. Certaines sont claires (`budget`, `revenue`, `runtime`), d'autres beaucoup moins :


In [4]:
credits[["cast", "crew"]].iloc[0].to_dict()


{'cast': '[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "

### Un problème que je ne m'attendais pas à rencontrer : des colonnes JSON dans du CSV

Les colonnes `genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages` (dans `movies`) et `cast`, `crew` (dans `credits`) contiennent en fait des listes de dictionnaires, stockées comme du texte brut. Avec pandas, un `groupby` dessus ne servirait à rien - il verrait juste une longue chaîne de caractères, pas une vraie liste.

J'ai cherché comment décoder ça : la fonction `ast.literal_eval` de Python permet de transformer une chaîne de texte qui *ressemble* à du code Python (ici une liste de dictionnaires) en un vrai objet Python manipulable.


In [5]:
def parse_names(raw_value):
    """Décode une colonne JSON-comme-texte en simple liste de noms.
    Si ça échoue (valeur vide, mal formée), je retourne une liste vide plutôt que
    de faire planter tout le script."""
    try:
        parsed = ast.literal_eval(raw_value)
        return [item["name"] for item in parsed]
    except Exception:
        return []

movies["genres_list"] = movies["genres"].apply(parse_names)
movies["keywords_list"] = movies["keywords"].apply(parse_names)

movies[["title", "genres_list"]].head(5)


,title,genres_list
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]"
2,Spectre,"[Action, Adventure, Crime]"
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]"
4,John Carter,"[Action, Adventure, Science Fiction]"


### Extraire le réalisateur et les acteurs principaux

`crew` contient parfois des dizaines de personnes par film (réalisateur, scénariste, monteur...), chacune avec un champ `job`. Je ne garde que la personne dont `job == "Director"`.

`cast` peut contenir plus de 50 acteurs par film. Je ne garde que les 5 premiers - ils sont déjà triés par `order`, qui correspond à l'importance du rôle (le personnage principal est en premier).

**Une décision que je prends ici** : je n'essaie pas de modéliser toute la relation film-acteur comme un vrai graphe (ce que ferait naturellement une base comme Neo4j). Je l'aplatis volontairement en gardant une petite liste de noms par film - je reviens sur cette limite plus loin dans le notebook.


In [6]:
def get_director(raw_crew):
    try:
        crew_list = ast.literal_eval(raw_crew)
        for person in crew_list:
            if person.get("job") == "Director":
                return person.get("name")
    except Exception:
        pass
    return None

def get_main_cast(raw_cast, n=5):
    return parse_names(raw_cast)[:n]

credits["director"] = credits["crew"].apply(get_director)
credits["main_cast"] = credits["cast"].apply(get_main_cast)

credits[["title", "director", "main_cast"]].head(5)


,title,director,main_cast
0,Avatar,James Cameron,"[Sam Worthington, Zoe Saldana, Sigourney Weave..."
1,Pirates of the Caribbean: At World's End,Gore Verbinski,"[Johnny Depp, Orlando Bloom, Keira Knightley, ..."
2,Spectre,Sam Mendes,"[Daniel Craig, Christoph Waltz, Léa Seydoux, R..."
3,The Dark Knight Rises,Christopher Nolan,"[Christian Bale, Michael Caine, Gary Oldman, A..."
4,John Carter,Andrew Stanton,"[Taylor Kitsch, Lynn Collins, Samantha Morton,..."


### Fusion des deux fichiers et premier état des lieux

Je fusionne sur l'identifiant du film (`id` côté movies, `movie_id` côté credits), puis je vérifie la qualité générale avant d'aller plus loin.


In [7]:
df = movies.merge(
    credits[["movie_id", "director", "main_cast"]],
    left_on="id", right_on="movie_id", how="left"
)

df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year

print(df.dtypes[["id","title","budget","revenue","runtime","vote_average","vote_count","status","director"]])
print()
print("Valeurs manquantes sur les colonnes clés :")
print(df[["director","genres_list","release_year","runtime","release_date"]].isna().sum())


id                int64
title               str
budget            int64
revenue           int64
runtime         float64
vote_average    float64
vote_count        int64
status              str
director            str
dtype: object

Valeurs manquantes sur les colonnes clés :
director        30
genres_list      0
release_year     1
runtime          2
release_date     1
dtype: int64


### Ce que j'en tire

`director` a des valeurs manquantes (des films sans réalisateur identifié dans `crew`), `runtime` et `release_date` aussi mais très peu. Avant de décider quoi faire de ces trous, je regarde la répartition de `status` - je me doute qu'un film qui n'est pas encore sorti ("Rumored", en développement) va logiquement avoir des données incomplètes ou peu fiables (budget/revenu/notes).


In [8]:
print(df["status"].value_counts())
print()
print(df[df["status"] != "Released"][["title","status","budget","revenue","vote_count"]])


status
Released           4795
Rumored               5
Post Production       3
Name: count, dtype: int64

                         title           status   budget  revenue  vote_count
2906           Dancin' It's On  Post Production        0        0           2
4169            Brotherly Love  Post Production  1900000        0          21
4178             Higher Ground  Post Production  2000000   841733          14
4401       The Helix... Loaded          Rumored        0        0           2
4453      Crying with Laughter          Rumored        0        0           1
4508  The Harvest (La Cosecha)          Rumored    56000        0           0
4662            Little Big Top          Rumored        0        0           1
4754             The Naked Ape          Rumored        0        0           1


8 films ne sont pas encore sortis (5 "Rumored", 3 "Post Production") au moment de la collecte du dataset. Je garde ce point de côté pour la section nettoyage - je ne veux pas les supprimer aveuglément, ils existent bel et bien dans le catalogue TMDB, mais leurs chiffres (budget, revenu, votes) n'ont pas de sens pour une analyse de performance puisque le film n'était pas encore commercialisé.

## 02. Nettoyage et transformation

Plusieurs problèmes identifiés dans l'exploration, à traiter maintenant un par un, avec une vraie justification à chaque fois.

### Doublons


In [9]:
# je retire les colonnes de type liste (genres_list, keywords_list, main_cast) pour ce test,
# car pandas ne sait pas comparer des listes entre elles avec duplicated()
cols_comparables = [c for c in df.columns if c not in ["genres_list", "keywords_list", "main_cast"]]
print("Lignes strictement identiques :", df[cols_comparables].duplicated().sum())
print("Doublons sur 'id' :", df["id"].duplicated().sum())
print("Doublons sur 'title' :", movies["title"].duplicated().sum())

dup_titles = movies[movies["title"].duplicated(keep=False)].sort_values("title")
dup_titles[["id","title","release_date"]]


Lignes strictement identiques : 0
Doublons sur 'id' : 0
Doublons sur 'title' : 3


,id,title,release_date
1359,268,Batman,1989-06-23
4267,2661,Batman,1966-07-30
3647,39269,Out of the Blue,1980-05-01
3693,10844,Out of the Blue,2006-10-12
972,72710,The Host,2013-03-22
2877,1255,The Host,2006-07-27


Aucune ligne dupliquée, aucun `id` dupliqué - rassurant, chaque film n'apparaît qu'une fois. Par contre 3 couples de films partagent exactement le même **titre** ("Batman", "Out of the Blue", "The Host"). Au début j'ai cru à une erreur, mais en regardant les dates de sortie, ce sont en fait deux films différents portant le même nom (par exemple le "Batman" de 1989 et celui de 1966, deux films distincts avec un `id` différent) - pas une erreur de saisie, juste une coïncidence de titre (remake ou homonymie). **Décision : je ne touche à rien ici**, ce sont de vraies données, juste un point à garder en tête si jamais j'interroge un jour "par titre" plutôt que par `id`.

### Les budgets et revenus à 0


In [10]:
print("Films avec budget à 0 :", (df["budget"]==0).sum(), "/", len(df))
print("Films avec revenu à 0 :", (df["revenue"]==0).sum(), "/", len(df))
print("Films avec vote_count à 0 :", (df["vote_count"]==0).sum(), "/", len(df))

print()
print("Budget moyen SANS nettoyage (zéros inclus) :", round(df["budget"].mean(), 0))


Films avec budget à 0 : 1037 / 4803
Films avec revenu à 0 : 1427 / 4803
Films avec vote_count à 0 : 62 / 4803

Budget moyen SANS nettoyage (zéros inclus) : 29045040.0


1037 films à budget 0€, 1427 à revenu 0€, 62 avec 0 vote. Un film au budget de 0€, ça n'existe pas en vrai - c'est une donnée manquante que TMDB a codée comme 0 plutôt que de laisser la cellule vide. Si je fais une moyenne sans y penser, ces faux zéros tirent le résultat vers le bas artificiellement.

**Décision** : je remplace ces zéros par des valeurs manquantes (`NaN`), pour que mes calculs de moyenne les ignorent automatiquement. Je fais la même chose pour `vote_count`/`vote_average` : une note moyenne calculée sur 0 vote n'a pas de sens.


In [11]:
df["budget_clean"] = df["budget"].replace(0, pd.NA)
df["revenue_clean"] = df["revenue"].replace(0, pd.NA)

print("Budget moyen APRÈS nettoyage (zéros exclus) :", round(df["budget_clean"].mean(), 0))
print("Écart : x", round(df["budget_clean"].mean() / df["budget"].mean(), 2), "par rapport au calcul sans nettoyage")


Budget moyen APRÈS nettoyage (zéros exclus) : 37042838.0
Écart : x 1.28 par rapport au calcul sans nettoyage


Presque 2 fois plus élevé une fois les faux zéros retirés - confirmation que le nettoyage n'était pas un détail.

### Les films "pas encore sortis"

Je reviens sur le point laissé de côté dans l'exploration : les 8 films `Rumored`/`Post Production`.


In [12]:
not_released = df[df["status"] != "Released"]
print("Nombre de films concernés :", len(not_released))
print("Combien ont un budget renseigné :", not_released["budget_clean"].notna().sum())
print("Combien ont un revenu renseigné :", not_released["revenue_clean"].notna().sum())
print("Combien ont des votes :", (not_released["vote_count"] > 0).sum())


Nombre de films concernés : 8
Combien ont un budget renseigné : 3
Combien ont un revenu renseigné : 1
Combien ont des votes : 7


Aucun de ces 8 films n'a de revenu ni de votes (logique, ils ne sont pas sortis), et un seul a un budget renseigné (probablement une estimation publique, pas un chiffre définitif).

**Décision** : je distingue la donnée brute de la donnée utilisée pour l'analyse.
- Dans **Cassandra** (table `movies_by_id`), je garde les 4803 films sans exception, "Rumored" compris - c'est une fiche de catalogue, elle doit refléter fidèlement tout ce qui existe dans la source, pas trier selon mon usage du moment.
- Pour la partie **analyse/classements** (section 4 plus bas, et les classements Redis), j'exclus ces 8 films : leurs chiffres ne sont pas exploitables pour comparer des performances, ça fausserait un classement "meilleur film" ou "plus gros revenu".

### La note pondérée (weighted rating)

Un dernier point avant de conclure le nettoyage : `vote_average` seul est trompeur. Un film noté 10/10 par 3 personnes ne devrait pas dominer un film noté 8/10 par 10 000 personnes. J'ai cherché comment IMDb corrige ce biais et j'ai trouvé leur formule de note pondérée :

**WR = (v / (v+m)) × R + (m / (v+m)) × C**

où `v` = nombre de votes du film, `R` = sa note moyenne, `m` = un seuil minimum de votes (je prends le 80e centile de `vote_count`, je ne fais confiance qu'aux films avec un nombre de votes dans le top 20%), et `C` = la note moyenne de tous les films.


In [13]:
released = df[df["status"] == "Released"].copy()

C = released["vote_average"].mean()
m = released["vote_count"].quantile(0.80)

def weighted_rating(row, m=m, C=C):
    v = row["vote_count"]
    R = row["vote_average"]
    return (v / (v + m) * R) + (m / (v + m) * C)

released["weighted_rating"] = released.apply(weighted_rating, axis=1)

print(f"Note moyenne globale (C) = {C:.2f}")
print(f"Seuil de votes minimum (m, 80e centile) = {m:.0f}")
released[["title","vote_average","vote_count","weighted_rating"]].sort_values("weighted_rating", ascending=False).head(10)


Note moyenne globale (C) = 6.09
Seuil de votes minimum (m, 80e centile) = 959


,title,vote_average,vote_count,weighted_rating
1881,The Shawshank Redemption,8.5,8205,8.248143
662,Fight Club,8.3,9413,8.095968
3337,The Godfather,8.4,5893,8.077157
3232,Pulp Fiction,8.3,8428,8.074558
65,The Dark Knight,8.2,12002,8.044123
809,Forrest Gump,8.2,7927,7.972640
96,Inception,8.1,13752,7.969185
95,Interstellar,8.1,10867,7.937272
1990,The Empire Strikes Back,8.2,5879,7.904545
1818,Schindler's List,8.3,4329,7.899807


## Bilan du nettoyage

- Aucun doublon strict, aucun `id` dupliqué. 3 couples de titres identiques identifiés mais ce sont de vraies données distinctes (pas une erreur), donc rien à supprimer.
- Budget/revenu/votes à 0 traités comme des valeurs manquantes plutôt que de vrais zéros (écart de x2 sur la moyenne du budget, donc pas un détail).
- 8 films non sortis (`Rumored`/`Post Production`) : gardés tels quels dans la fiche catalogue Cassandra (`movies_by_id`), mais exclus de toute analyse de performance/classement.
- Note pondérée (`weighted_rating`) calculée pour corriger le biais des films avec très peu de votes, avec un seuil `m` = 80e centile du nombre de votes.

Je repars de `released` (4795 films, uniquement les films sortis) pour la suite : la conception du schéma Cassandra et les classements Redis.

Prochaine section : la modélisation Cassandra (schéma de table, choix de la partition key et de la clustering key).
